# CASMI 2026: Molecule Identification from Mass Spectra
### High-Precision Multi-Collision Logit Retrieval across 717k Unified Natural Products Pool

**Islamic University of Science & Technology (IUST), Kashmir**  
*Department of Computer Science & Engineering · Centre for Artificial Intelligence (CAI)*

---

### 👥 Research Team & Authorship
| Role | Name | Designation & Affiliation | Research & Project Contribution |
| :--- | :--- | :--- | :--- |
| **Faculty Research Supervisor** | **Dr. Asif Assad*** | Senior Assistant Professor & Faculty Supervisor, Dept. of CSE, IUST | Project Supervision, Research Direction, Theoretical Review & Methodological Oversight |
| **Lead Author & Principal Researcher** | **Dar Rehan Rasool** | M.Tech Student, Centre for Artificial Intelligence, IUST | End-to-End System Architecture, SpectrumFPNet Model, Out-of-Core Big-Data Streaming, Candidate Retrieval & Dashboard |
| **Contributing Researcher** | **Muneeb Syed Shah** | M.Tech Student, Centre for Artificial Intelligence, IUST | Mass Spectrometry Data Preprocessing, Adduct Formulation, Feature Engineering |
| **Contributing Researcher** | **Shafiya Bashir Pampori** | M.Tech Student, Centre for Artificial Intelligence, IUST | Chemical Candidate Filtering, Tanimoto Verification & Statistical Benchmark Validation |

---

### 📊 Validated Model Benchmarks (60-Epoch University GPU Run)
| Evaluation Metric | Validation Score | Benchmark Interpretation |
| :--- | :---: | :--- |
| **Candidate Pool Size** | **717,156 Structures** | Unified COCONUT 2.0 + CASMI Training Set (Breaks 275k coverage barrier) |
| **Bit-Level Accuracy** | **98.15%** | Highly stable 2048-bit global classification |
| **ROC-AUC Score** | **87.40%** | Robust discrimination across all 2048 Morgan bits |
| **Active-Bit Precision** | **84.10%** | 10,376 True Positives vs. only 1,961 False Positives |
| **Average Precision (AP)** | **42.98%** | **17.2× higher** than the 2.5% random baseline |
| **Mass Filter Calibration** | **+1.5 ppm / ±8.5 ppm** | Calibrated Bruker timsTOF physical drift & adaptive window |
| **Deduplication Strategy** | **InChIKey14** | Eliminates stereoisomers to maximize Top-25 planar skeleton entropy |

> **Dynamic Kaggle Submission**: Dynamically vectorizes `test.parquet` and aligns with `sample_submission.csv` to ensure 100% compliance with Kaggle hidden test sets.


In [ ]:
# Cell 1: Environment & Dependency Setup
import os, sys, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'[*] Active Hardware Device: {device}')
if torch.cuda.is_available():
    print(f'[+] GPU: {torch.cuda.get_device_name(0)}')

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# Cell 2: Neural Architecture & Spectrum Vectorization
class SpectrumFPNet(nn.Module):
    def __init__(self, in_dim: int = 12001, out_dim: int = 2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(1024, out_dim),
        )
    def forward(self, x):
        return self.net(x)

def bin_spectrum(mzs, intensities, n_bins=12000, bin_size=0.1):
    out = np.zeros(n_bins, dtype=np.float32)
    if mzs is None or intensities is None or len(mzs) == 0:
        return out
    mzs = np.asarray(mzs, dtype=np.float32)
    intensities = np.asarray(intensities, dtype=np.float32)
    n = min(len(mzs), len(intensities))
    if n == 0: return out
    mzs, intensities = mzs[:n], intensities[:n]
    valid = np.isfinite(mzs) & np.isfinite(intensities) & (mzs >= 0) & (intensities > 0)
    if not np.any(valid): return out
    mzs, intensities = mzs[valid], intensities[valid]
    intensities = np.sqrt(intensities)
    idx = np.floor(mzs / bin_size).astype(np.int64)
    valid_bins = (idx >= 0) & (idx < n_bins)
    if not np.any(valid_bins): return out
    np.add.at(out, idx[valid_bins], intensities[valid_bins])
    m = np.max(out)
    if m > 0: out /= m
    return out

PROTON = 1.007276
ADDUCT_DELTA = {
    '[M+H]+': PROTON,
    '[M+NH4]+': PROTON + 17.026549,
    '[M-H2O+H]+': PROTON - 18.010565,
    '[M-2H2O+H]+': PROTON - 2 * 18.010565,
    '[M+Na]+': 22.989218,
    '[M+K]+': 38.963158,
    '[M-H]-': -PROTON,
    '[M-H2O-H]-': -PROTON - 18.010565,
    '[M+CH2O2-H]-': -PROTON + 46.00548,
    '[M+Cl]-': 34.968853,
}

def estimate_neutral_mass(precursor_mz, adduct):
    delta = ADDUCT_DELTA.get(adduct)
    return (precursor_mz - delta) if delta is not None else None


In [ ]:
# Cell 3: Locate Files & Load Model + 717k Candidate Pool
def find_file(name, candidate_paths=[]):
    for p in candidate_paths:
        f = Path(p) / name
        if f.is_file(): return f
    if Path("/kaggle/input").exists():
        for f in Path("/kaggle/input").rglob(name):
            if f.is_file(): return f
    return None

search_dirs = [
    ".", "..", "models", "data/processed", "kaggle_upload",
    "/kaggle/input", "/kaggle/working"
]

model_path = find_file("best_model.pt", search_dirs)
# Look for unified candidate pool, with fallback to fp_database.npz
fp_db_path = find_file("candidate_pool.npz", search_dirs)
if not fp_db_path:
    fp_db_path = find_file("fp_database.npz", search_dirs)
mass_path = find_file("candidate_masses.npy", search_dirs)

print(f"[+] Model Checkpoint:   {model_path}")
print(f"[+] Candidate Database: {fp_db_path}")
print(f"[+] Candidate Masses:   {mass_path}")

assert model_path is not None, "best_model.pt not found! Please attach dataset."
assert fp_db_path is not None, "Candidate database not found! Please attach dataset."

ckpt = torch.load(model_path, map_location=device, weights_only=False)
model = SpectrumFPNet(ckpt["in_dim"], 2048).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"[+] Successfully loaded SpectrumFPNet model on {device}")

with np.load(fp_db_path, allow_pickle=True) as data:
    CANDIDATE_FPS = data["fps"]
    CANDIDATE_SMILES = data["smiles"]
    CANDIDATE_INCHIKEY14 = data["inchikey14"]
    CANDIDATE_MASSES = data["masses"] if "masses" in data else (np.load(mass_path) if mass_path else None)

assert CANDIDATE_MASSES is not None, "Candidate exact masses missing!"
print(f"[+] Loaded {len(CANDIDATE_SMILES):,} candidate molecules with exact neutral masses.")


In [ ]:
# Cell 4: Load Dynamic Competition Test Data
# In Kaggle competitions, test.parquet is dynamically swapped during submission re-run
test_parquet_path = find_file('test.parquet', ['data/raw', '/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra', '/kaggle/input'])
sample_sub_path = find_file('sample_submission.csv', ['data/raw', '/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra', '/kaggle/input'])

print(f'[+] Dynamic Test Parquet: {test_parquet_path}')
print(f'[+] Sample Submission: {sample_sub_path}')

test_df = pd.read_parquet(test_parquet_path)
sample_sub = pd.read_csv(sample_sub_path) if sample_sub_path else None

print(f'[+] Loaded {len(test_df):,} test spectra across {test_df["molecule_id"].nunique():,} unique molecules.')
if sample_sub is not None:
    print(f'[+] Competition requires exactly {len(sample_sub):,} submission rows.')


In [ ]:
# Cell 5: Dynamic Spectrum Vectorization & Batch Neural Inference
print("[*] Vectorizing test spectra into 12,000 bins...")
spectra_list = [bin_spectrum(m, i) for m, i in zip(test_df["ms2_mzs"], test_df["ms2_normalized_intensities"])]
test_spectra = np.stack(spectra_list).astype(np.float32)
test_pmzs = test_df["precursor_mz"].values.astype(np.float32)
test_adducts = test_df["adduct"].values
test_mol_ids = test_df["molecule_id"].values

X_test = np.concatenate([
    test_spectra,
    (test_pmzs / 1000.0).reshape(-1, 1).astype(np.float32)
], axis=1)

raw_logits = []
batch_size = 256
with torch.no_grad():
    for i in tqdm(range(0, len(X_test), batch_size), desc="Neural Inference"):
        xb = torch.from_numpy(X_test[i:i + batch_size]).to(device)
        logits = model(xb).cpu().numpy()
        raw_logits.append(logits)
raw_logits = np.concatenate(raw_logits, axis=0)

# Group spectra by molecule_id and perform Multi-Collision Energy Logit Fusion
mol_logits_map = defaultdict(list)
mol_meta_map = {}
for i, mid in enumerate(test_mol_ids):
    mol_logits_map[mid].append(raw_logits[i])
    mol_meta_map[mid] = (test_pmzs[i], test_adducts[i])

fused_query_logits = {}
fused_query_fps = {}
for mid, l_list in mol_logits_map.items():
    # Average unconstrained logits across collision energies (fuses fragmentation regimes)
    f_logit = np.mean(np.stack(l_list), axis=0)
    fused_query_logits[mid] = f_logit
    fused_query_fps[mid] = 1.0 / (1.0 + np.exp(-f_logit))

print(f"[+] Completed Multi-Collision Logit Fusion for {len(fused_query_logits):,} unique test molecules.")


In [ ]:
# Cell 6: Sub-ppm Drift Calibration, Fast Binary Retrieval & InChIKey14 Deduplication
def retrieve_top25(query_fp, query_logit, precursor_mz, adduct, top_k=25):
    neutral = estimate_neutral_mass(precursor_mz, adduct)
    if neutral is not None:
        # Bruker timsTOF +1.5 ppm systematic drift calibration
        neutral_cal = neutral * (1.0 + 1.5e-6)
    else:
        neutral_cal = None
    
    candidate_indices = None
    if neutral_cal is not None and CANDIDATE_MASSES is not None:
        # Adaptive mass tolerance: 8.5 ppm -> 15.0 ppm -> 25.0 ppm
        for ppm_tol in [8.5, 15.0, 25.0]:
            tol = neutral_cal * (ppm_tol * 1e-6)
            left = np.searchsorted(CANDIDATE_MASSES, neutral_cal - tol, side="left")
            right = np.searchsorted(CANDIDATE_MASSES, neutral_cal + tol, side="right")
            if right - left >= top_k:
                candidate_indices = np.arange(left, right)
                break
        if candidate_indices is None or len(candidate_indices) < top_k:
            # Fallback to closest 100
            diffs = np.abs(CANDIDATE_MASSES - neutral_cal)
            candidate_indices = np.argsort(diffs)[:100]
    
    if candidate_indices is not None and len(candidate_indices) > 0:
        sub_fps = CANDIDATE_FPS[candidate_indices].astype(np.float32)
        bit_counts = sub_fps.sum(axis=1)
        
        # Normalized logit dot product (rewards positive substructures, penalizes false positives)
        logit_scores = (sub_fps @ query_logit) / np.sqrt(bit_counts + 5.0)
        
        # Continuous Tanimoto on probabilities
        inter = sub_fps @ query_fp
        union = bit_counts + query_fp.sum() - inter + 1e-9
        tanimoto_scores = inter / union
        
        # Dual rank fusion
        norm_logit = (logit_scores - logit_scores.mean()) / (logit_scores.std() + 1e-6)
        norm_tani = (tanimoto_scores - tanimoto_scores.mean()) / (tanimoto_scores.std() + 1e-6)
        combined_scores = 0.5 * norm_logit + 0.5 * norm_tani
        
        order = np.argsort(-combined_scores)
        selected_indices = candidate_indices[order]
    else:
        # Global fallback if neutral mass could not be determined
        inter = CANDIDATE_FPS.astype(np.float32) @ query_fp
        union = CANDIDATE_FPS.sum(1) + query_fp.sum() - inter + 1e-9
        sims = inter / union
        selected_indices = np.argsort(-sims)[:top_k * 5]
    
    # Planar Skeleton Deduplication (InChIKey14)
    seen_ik = set()
    out_smi = []
    for idx in selected_indices:
        k = CANDIDATE_INCHIKEY14[idx]
        if k in seen_ik: continue
        seen_ik.add(k)
        out_smi.append(str(CANDIDATE_SMILES[idx]))
        if len(out_smi) >= top_k: break
    
    while len(out_smi) < top_k:
        out_smi.append(out_smi[-1] if out_smi else "CCO")
    return out_smi

print("[*] Executing calibrated candidate retrieval across 717k natural product pool...")
pred_map = {}
for mid in tqdm(fused_query_logits.keys(), desc="Candidate Retrieval"):
    q_logit = fused_query_logits[mid]
    q_fp = fused_query_fps[mid]
    pm, ad = mol_meta_map[mid]
    cands = retrieve_top25(q_fp, q_logit, pm, ad, top_k=25)
    pred_map[mid] = ";".join(cands[:25])

print(f"[+] Completed candidate retrieval for {len(pred_map):,} test molecules.")


In [ ]:
# Cell 7: Align with sample_submission.csv & Save Zero-Defect submission.csv
if sample_sub is not None:
    submission = sample_sub[['molecule_id']].copy()
    submission['smiles'] = submission['molecule_id'].map(pred_map)
    fallback_smi = ';'.join(['CCO'] * 25)
    submission['smiles'] = submission['smiles'].fillna(fallback_smi)
else:
    submission = pd.DataFrame([
        {'molecule_id': mid, 'smiles': smi} for mid, smi in pred_map.items()
    ])

submission.to_csv('submission.csv', index=False)
print('=== Final Submission Verification ===')
print(f'Shape: {submission.shape}')
assert list(submission.columns) == ['molecule_id', 'smiles'], 'Columns must be molecule_id,smiles'
assert submission['molecule_id'].isnull().sum() == 0, 'Null molecule_ids found!'
assert submission['smiles'].isnull().sum() == 0, 'Null smiles found!'
assert submission['molecule_id'].duplicated().sum() == 0, 'Duplicate molecule_ids found!'
cand_counts = submission['smiles'].apply(lambda s: len(s.split(';')))
assert (cand_counts == 25).all(), 'Every row must have exactly 25 candidates!'
print('[+] Zero-defect submission.csv written successfully to /kaggle/working/submission.csv!')
print(submission.head(3))
